# Lekcija 02 - Raziskovanje ogrodja Microsoft Agent

**Microsoft Agent Framework (MAF)** je enotno ogrodje za izdelavo AI agentov. Ponuja čisto, sestavljivo arhitekturo s štirimi osnovnimi gradniki:

- **Client** – se poveže s končno točko AI modela in upravlja komunikacijo
- **Agent** – ovije klienta z navodili in definicijami orodij
- **Tools** – razširijo zmožnosti agenta s prilagojenimi funkcijami, ki jih model lahko kliče
- **Session** – ohranja zgodovino pogovora za interakcije z več hodi

V tej lekciji bomo zgradili **agenta za rezervacijo potovanj**, ki preverja razpoložljivost destinacij z uporabo teh konceptov.


## Namestitev


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Razumevanje arhitekture Agent Frameworka

Microsoft Agent Framework sledi slojeviti arhitekturi:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Odjemalec** – `FoundryChatClient` se poveže z Azure OpenAI namestitvijo. Upravlja preverjanje pristnosti, oblikovanje zahtevka in razčlenjevanje odziva.
2. **Agent** – Ustvari se iz odjemalca preko `provider.create_agent()`, agent združuje dostop do modela z navodili (sistemski poziv) in orodji.
3. **Orodja** – Python funkcije, okrašene z `@tool`, ki jih lahko agent pokliče za izvajanje dejanj ali pridobivanje podatkov.
4. **Seansa** – Objekt `AgentSession` (ustvarjen preko `agent.create_session()`), ki hrani zgodovino pogovora in omogoča večkrožni dialog, kjer se agent spomni preteklega konteksta.

Zgradimo vsak sloj korak za korakom.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Dodajanje orodij z dekoratorjem @tool

Orodja omogočajo agentom izvajanje dejanj, ki gredo onkraj generiranja besedila. Dekorator `@tool` spremeni običajno Python funkcijo v nekaj, kar agent lahko pokliče.

Ključne točke:
- Uporabite `Annotated[type, "description"]`, da model razume vsak parameter.
- Dokumentacijska vrstica postane opis orodja, ki ga model vidi.
- `approval_mode="never_require"` pomeni, da se orodje izvede samodejno brez potrditve uporabnika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Ustvarjanje agenta z orodji

Zdaj združimo odjemalca, navodila in orodja v agenta. `instructions` delujejo kot sistemski poziv — določajo persono in vedenje agenta.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Večkrožni pogovori s seansami

`AgentSession` (ustvarjen prek `agent.create_session()`) beleži vsa sporočila v pogovoru. Z uporabo iste seje pri vsakem klicu `agent.run()` ima agent dostop do celotne zgodovine pogovora in se lahko sklicuje na prejšnja sporočila.

Podamo `tools=[check_destination_availability]`, da lahko agent ob vsakem potezu pokliče našega preverjevalca razpoložljivosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Povzetek

V tej lekciji ste raziskali štiri stebre Microsoft Agent Frameworka:

| Koncept | Kaj ste se naučili |
|---------|------------------|
| **Odjemalec** | `FoundryChatClient` se poveže z Azure OpenAI z avtentikacijo na podlagi poverilnic |
| **Agent** | `provider.create_agent()` poveže model z navodili in imenom |
| **Orodja** | Dekorator `@tool` omogoča klic Python funkcij agentu |
| **Seja** | `agent.create_session()` ohranja zgodovino pogovora skozi več potez |

Ti gradniki skupaj sestavijo agente, ki lahko vodijo naravne pogovore, kličejo zunanje funkcije in ohranjajo kontekst — temelj za bolj napredne agentne vzorce v nadaljnjih lekcijah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
